# 06 â€” Calcul des embeddings BGE-M3

Encode les chunks enfants en vecteurs multilingues normalisÃ©s et vÃ©rifie dimensions, normes et temps d'encodage.

> VÃ©rifiez toujours les drapeaux `RUN_*` avant l'exÃ©cution. Pour cette reconstruction propre, les traitements locaux nÃ©cessaires des Ã©tapes 02 Ã  07 sont dÃ©jÃ  activÃ©s ; les collectes rÃ©seau et Ã©valuations GPU facultatives restent dÃ©sactivÃ©es.


In [1]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'config.yaml').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Impossible de trouver config.yaml et src/.')

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('Projet :', PROJECT_ROOT)
print('Python :', sys.executable)
print('DonnÃ©es actives :', PROJECT_ROOT / 'data')


Projet : <PROJECT_ROOT>
Python : <PROJECT_ROOT>\.venv\Scripts\python.exe
DonnÃ©es actives : <PROJECT_ROOT>\data


In [2]:
import json
import time
import numpy as np
import pandas as pd
from chunker import Chunk
from config import load_config

config = load_config(PROJECT_ROOT / 'config.yaml')
chunk_path = config.project.data_path / 'chunks' / 'chunks.jsonl'
chunks = [Chunk(**json.loads(line)) for line in chunk_path.read_text(encoding='utf-8').splitlines() if line.strip()]
economic_path = config.local_documents.chunks_full_path
economic_chunks = [Chunk(**json.loads(line)) for line in economic_path.read_text(encoding='utf-8').splitlines() if line.strip()]
facebook_path = config.facebook_scraping.chunks_candidate_full_path
facebook_chunks = [
    Chunk(**json.loads(line))
    for line in facebook_path.read_text(encoding='utf-8').splitlines()
    if line.strip()
] if facebook_path.exists() else []
print('ModÃ¨le :', config.embeddings.model_name)
print('Chunks ENSEA :', len(chunks), '| Ã©conomiques :', len(economic_chunks))
print('Chunks Facebook :', len(facebook_chunks), '| dimension attendue : 1024 | prÃ©cision :', config.embeddings.precision)


ModÃ¨le : BAAI/bge-m3
Chunks ENSEA : 4719 | Ã©conomiques : 52486
Chunks Facebook : 293 | dimension attendue : 1024 | prÃ©cision : fp16


In [3]:
from embeddings import EmbeddingModel

RUN_SAMPLE_EMBEDDING = False
if RUN_SAMPLE_EMBEDDING:
    model = EmbeddingModel(config.embeddings)
    sample_texts = [chunk.text for chunk in chunks[:16]]
    started = time.perf_counter()
    sample_vectors = model.encode(sample_texts)
    elapsed = time.perf_counter() - started
    display(pd.Series({
        'forme': str(sample_vectors.shape),
        'secondes': elapsed,
        'norme_moyenne': float(np.linalg.norm(sample_vectors, axis=1).mean()),
        'mÃ©moire_Mo': sample_vectors.nbytes / 1024**2,
    }).to_frame('valeur'))
else:
    print('Encodage GPU dÃ©sactivÃ©. Passez RUN_SAMPLE_EMBEDDING Ã  True pour tester 16 chunks.')


Encodage GPU dÃ©sactivÃ©. Passez RUN_SAMPLE_EMBEDDING Ã  True pour tester 16 chunks.


In [4]:
RUN_EMBED_ALL = True
embedding_path = config.project.data_path / 'embeddings' / 'document_vectors.npy'
economic_embedding_path = config.local_documents.vectors_full_path
facebook_embedding_path = config.project.data_path / 'facebook' / 'vectors.npy'
if RUN_EMBED_ALL:
    if 'model' not in globals():
        model = EmbeddingModel(config.embeddings)
    def encode_chunks(items):
        batches = []
        for start in range(0, len(items), config.embeddings.batch_size):
            texts = [chunk.text for chunk in items[start:start + config.embeddings.batch_size]]
            batches.append(model.encode(texts))
        return np.concatenate(batches, axis=0)

    started = time.perf_counter()
    embedding_path.parent.mkdir(parents=True, exist_ok=True)
    vectors = encode_chunks(chunks)
    np.save(embedding_path, vectors)
    print('Matrice ENSEA :', vectors.shape)
    economic_embedding_path.parent.mkdir(parents=True, exist_ok=True)
    economic_vectors = encode_chunks(economic_chunks)
    np.save(economic_embedding_path, economic_vectors)
    print('Matrice Ã©conomique :', economic_vectors.shape)
    if facebook_chunks:
        facebook_embedding_path.parent.mkdir(parents=True, exist_ok=True)
        facebook_vectors = encode_chunks(facebook_chunks)
        np.save(facebook_embedding_path, facebook_vectors)
        print('Matrice Facebook :', facebook_vectors.shape)
    print('Temps total :', round(time.perf_counter() - started, 2), 's')
    model.unload()
else:
    print('Encodage complet dÃ©sactivÃ© ; sortie prÃ©vue :', embedding_path)


You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Matrice ENSEA : (4719, 1024)


Matrice Ã©conomique : (52486, 1024)


Matrice Facebook : (293, 1024)
Temps total : 376.75 s


## ParticularitÃ© de BGE-M3

BGE-M3 est multilingue, comprend bien le franÃ§ais et produit des vecteurs de 1 024 dimensions. La normalisation permet d'utiliser directement la similaritÃ© cosinus. Les embeddings dÃ©crivent les chunks enfants, pas les parents.


## Ce qui est rÃ©ellement encodÃ©

BGE-M3 encode sans distinction arbitraire les passages narratifs et les faits tabulaires. Pour ces
derniers, le texte vectorisÃ© contient explicitement l'indicateur, la pÃ©riode, la valeur, l'unitÃ© et
le domaine. BM25 conservera parallÃ¨lement les mÃªmes facettes Ã  l'Ã©tape suivante.

In [5]:
all_embedding_chunks = chunks + economic_chunks + facebook_chunks
embedding_chunk_df = pd.DataFrame([chunk.__dict__ for chunk in all_embedding_chunks])
display(embedding_chunk_df.groupby(['category', 'content_type']).size().rename('chunks Ã  encoder').to_frame())
table_examples = embedding_chunk_df[embedding_chunk_df['content_type'].eq('table_fact')]
malformed_indicator_count = int((~table_examples['indicator'].map(
    lambda value: any(char.isalpha() for char in str(value))
)).sum()) if not table_examples.empty else 0
display(table_examples[['indicator', 'period', 'value', 'unit', 'page', 'text']].head(10))
assert not table_examples.empty
assert malformed_indicator_count == 0, 'Refus dâ€™encoder des indicateurs purement numÃ©riques.'
if RUN_EMBED_ALL:
    assert len(vectors) == len(chunks)
    assert len(economic_vectors) == len(economic_chunks)
    if facebook_chunks:
        assert len(facebook_vectors) == len(facebook_chunks)
    print('Toutes les lignes, y compris les faits tabulaires, ont un vecteur alignÃ©.')

,,chunks Ã encoder
category,content_type,
%f0%9d%90%80%f0%9d%90%a9%f0%9d%90%a9%f0%9d%90%9e%f0%9d%90%a5-a-%f0%9d%90%a9%f0%9d%90%ab%f0%9d%90%a8%f0%9d%90%a3%f0%9d%90%9e%f0%9d%90%ad%f0%9d%90%ac-%f0%9d%90%9d%f0%9d%90%9e-%f0%9d%90%ab,text,1
%f0%9d%90%82%f0%9d%90%8e%f0%9d%90%8d%f0%9d%90%82%f0%9d%90%8e%f0%9d%90%94%f0%9d%90%91%f0%9d%90%92-%f0%9d%90%83%f0%9d%90%84%f0%9d%90%8d%f0%9d%90%93%f0%9d%90%91%f0%9d%90%84%f0%9d%90%84,text,1
%f0%9d%90%82%f0%9d%90%a8%f0%9d%90%a7%f0%9d%90%9c%f0%9d%90%a8%f0%9d%90%ae%f0%9d%90%ab%f0%9d%90%ac-%f0%9d%90%9f%f0%9d%90%a2%f0%9d%90%a7%f0%9d%90%9a%f0%9d%90%a5-%f0%9d%90%80%f0%9d%90%92-%f0%9d%9f%90,text,1
%f0%9d%90%83%f0%9d%90%8e%f0%9d%90%94%f0%9d%90%81%f0%9d%90%8b%f0%9d%90%84-%f0%9d%90%82%f0%9d%90%84%f0%9d%90%91%f0%9d%90%84%f0%9d%90%8c%f0%9d%90%8e%f0%9d%90%8d%f0%9d%90%88%f0%9d%90%84-%f0%9d%90%83,text,5
%f0%9d%90%84%f0%9d%90%8d%f0%9d%90%92%f0%9d%90%84%f0%9d%90%80%f0%9d%90%81%f0%9d%90%82%f0%9d%90%84%f0%9d%90%80%f0%9d%90%8e-%f0%9d%90%82%f0%9d%90%8e%f0%9d%90%85%f0%9d%90%84%f0%9d%90%81-%f0%9d%90%91,text,2
...,...,...
visite-des-enseignants-de-michigan-state-university-a-yamoussoukro,text,1
visite-du-nouveau-directeur-de-la-dgri-au-cea-ensea-dabidjan,text,1
visite-dune-delegation-de-lins-de-la-guinee-conakry-a-lensea-dabidjan,text,3


,indicator,period,value,unit,page,text
4733,Enseignement,décembre-20,"0,1",%,2.0,Fait statistique exact | Indicateur : Enseigne...
4734,Restaurants et Hôtels,décembre-21,"0,6",%,2.0,Fait statistique exact | Indicateur : Restaura...
4735,Biens et services divers,décembre-21,"0,5",%,2.0,Fait statistique exact | Indicateur : Biens et...
4736,"INDICE GLOBAL -2,0 -1,0",décembre-20,"0,0",%,2.0,Fait statistique exact | Indicateur : INDICE G...
4737,"INDICE GLOBAL -2,0 -1,0",décembre-21,"1,0",%,2.0,Fait statistique exact | Indicateur : INDICE G...
4738,"Produits alimentaires, boissons non alcoolisées",décembre-19,"0,1",%,2.0,Fait statistique exact | Indicateur : Produits...
4739,"Boissons alcoolisées, tabacs et stupéfiants",décembre-20,"0,2",%,2.0,Fait statistique exact | Indicateur : Boissons...
4740,"Articles d'habillement et chaussures Logement,...",décembre-20,"0,1",%,2.0,Fait statistique exact | Indicateur : Articles...
4741,"Articles d'habillement et chaussures Logement,...",décembre-21,"0,9",%,2.0,Fait statistique exact | Indicateur : Articles...
4742,"Meubles, articles de ménage et entretien coura...",décembre-20,"0,3",%,2.0,"Fait statistique exact | Indicateur : Meubles,..."


Toutes les lignes, y compris les faits tabulaires, ont un vecteur alignÃ©.
